In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import hydra
import lightning as lit
import torch
import torch.nn as nn
import wandb
import palettable
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from lightning.pytorch.callbacks import LearningRateMonitor, EarlyStopping, RichProgressBar, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger, CSVLogger
from loguru import logger
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from datetime import datetime

from lightning.pytorch.loggers import WandbLogger, CSVLogger

from machine_intelligence.models.GatedTransformer import GatedTransformer
from machine_intelligence.utilities.data_handling import load_parquet_table
from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule
from machine_intelligence.data.TransformsData import ZeroFill

### Obsolete ! Use the performance_eval.py script

In [ ]:
torch.use_deterministic_algorithms(True)


In [ ]:

cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime('%Y-%m-%d')
plot_output_dir = project_root / 'plot_outputs' / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:

# model_path = project_root / Path("experiment_logs/gtn-final/version_20/checkpoints/last.ckpt")
# model_path = project_root / Path("experiment_logs/gtn-final/version_19/checkpoints/epoch=35-step=288.ckpt")
model_hparams_path = project_root / Path("experiment_logs/gtn-final/version_15/hparams.yaml")



# model = GatedTransformer.load_from_checkpoint(model_path)
with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml",
                  overrides=["model=gtn", "optimizer=Adam_gtn", "data=cohort_a_r0.4_prev_0.3_imp_none",
                             "params.batch_size_gpu=256", "experiment.experiment_name=gtn-final-test",
                             "experiment.dev_run=False", "experiment.run_test=True", "experiment.tracked_run=True",
                             "experiment.n_workers=12", "experiment.profiling=False"], return_hydra_config=True)
    # cfg.data.directories = "${hydra.runtime.cwd}/datasets/sepsis_type_a_cohort"

    print(cfg)



In [ ]:

# lit.seed_everything(seed=42, workers=True)

# model = GatedTransformer.load_from_checkpoint(checkpoint_path=model_path)

device: torch.device = (
    torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
)
if torch.cuda.is_available():
    # torch.cuda.manual_seed(42)
    # torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:

wandb.init(
    # set the wandb project where this run will be logged
    project=cfg.experiment.experiment_name,
    # track hyperparameters and run metadata
)
# this enables code saving make sure you don't save data by accident as this might flood the free storage plan
wandb_logger = WandbLogger(
    save_dir="experiment_logs",
    project=cfg.experiment.experiment_name,
)

csv_logger = CSVLogger("experiment_logs", name=cfg.experiment.experiment_name)


In [ ]:

# wandb.init(mode="disabled")
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)


In [ ]:
trainer = lit.Trainer(
    deterministic=True,
    devices=1,
    # accelerator="cpu",
    benchmark=False,
    logger=[csv_logger, wandb_logger],

)



test_result = trainer.test(
        model=model, datamodule=sepsis_pred_data, verbose=True
    )


In [ ]:
# model = GatedTransformer.load_from_checkpoint(checkpoint_path=model_path)

In [ ]:
sepsis_pred_data.setup("test")

test_dl = sepsis_pred_data.test_dataloader()
# trainer.test(model, dataloaders=test_dl, ckpt_path=model_path, verbose=True)

model_path = project_root / Path("machine_intelligence/models/trained_models/gtn_final/checkpoints/last.ckpt")
model = GatedTransformer.load_from_checkpoint(checkpoint_path=model_path)

# model.eval()
# print(model.parameters())

logger.info(f"Pre Test: {model.classifier_head[6].weight}")


with logger.catch():
    # trainer.test(model, datamodule=sepsis_pred_data)
    # trainer.test(model,datamodule=sepsis_pred_data, ckpt_path=model_path)
   trainer.test(model, dataloaders=test_dl)


logger.info(f"After Test: {model.classifier_head[6].weight}")
    
